# TBD Phase 2 26L: Performance & Computing Models

## Introduction

In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [13]:
GROUP_ID = 10
NOTEBOOK_URL = "https://github.com/jarrok3/tbd-workshop/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Jakub Najdek / 323063",
    "Viktoriia Nowotka / 347990",
    "Jarosław Jaworski / 342189",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [14]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.2 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: faker
    Found existing installation: Faker 40.21.0
    Uninstalling Faker-40.21.0:
      Successfully uninstalled Faker-40.21.0

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.security.manager=allow"

import gc
import time
import json
import tracemalloc
import statistics
from datetime import date
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from pyspark import SparkContext
from memory_profiler import memory_usage
from pyspark.sql import SparkSession
from IPython.display import Markdown, display

In [16]:
print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.13.2
Polars: 1.41.2
Pandas: 3.0.3
DuckDB: 1.5.3
CPU logical cores: 10
RAM GiB: 16.0


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


### Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [24]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Web analytics',
 'feature': 'referrers',
 'stress': 'funnel-like aggregation'}

### Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [25]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
SCALE = "medium"
SCALE_ROWS = {
    "debug": 20_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 10 {'name': 'Web analytics', 'feature': 'referrers', 'stress': 'funnel-like aggregation'}
Rows: 10000000
Run seed recorded in manifest: 294203240347823298695370432566668146720
Output directory: ../data/phase2_26L/group_10


### Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [8]:
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))

base_urls   = ["https://example.com/home", "https://example.com/product",
                   "https://example.com/checkout", "https://example.com/blog",
                   "https://example.com/about"]
page_titles = ["Home", "Product Page", "Checkout", "Blog", "About Us"]

def customize_for_variant(df, card, rng):
    n = len(df)

    # rare category (Linux)
    os_options = ["Windows", "macOS", "Android", "iOS", "Linux"]
    os_weights = [0.30, 0.15, 0.35, 0.19, 0.01]
    url_idx = rng.choice(len(base_urls), size=n)

    df = df.with_columns([
        pl.Series("operating_system", np.array(os_options)[rng.choice(len(os_options), size=n, p=os_weights)]),
        pl.Series("page_url", np.array(base_urls)[url_idx]),
        pl.Series("page_title", np.array(page_titles)[url_idx]),
    ])

    df = df.rename({"event_id": "session_id", "entity_id": "visitor_id", "event_ts": "entry_timestamp", "category": "browser", "metric_1": "time_spent", "metric_2": "ads_viewed"})

    # nulls
    mask = pl.Series(rng.random(n) < 0.01)
    df = df.with_columns( pl.when(mask).then(None).otherwise(pl.col("ads_viewed")).alias("ads_viewed") )
    df = df.with_columns( pl.when(mask).then(None).otherwise(pl.col("browser")).alias("browser"))
    df = df.with_columns( pl.when(mask).then(None).otherwise(pl.col("operating_system")).alias("operating_system") )

    return df


def generate_dimension_table(card, rng):
    return pl.DataFrame(
        {
            "page_url": base_urls,
            "page_title": page_titles,
            "page_section": ["marketing", "catalog", "sales", "marketing", "info"],
            "requires_auth": [False, False, True, False, False],
        }
    )

In [9]:
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
events.sort(["entry_timestamp", "browser"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=10_000,
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


{
  "created_at_utc": "2026-06-08T08:42:30.812381+00:00",
  "group_id": 10,
  "variant": {
    "name": "Web analytics",
    "feature": "referrers",
    "stress": "funnel-like aggregation"
  },
  "scale": "medium",
  "rows": 10000000,
  "run_seed": 185816500262043428825167801116489674646,
  "paths": {
    "events": "..\\data\\phase2_26L\\group_10\\events.parquet",
    "events_partitioned": "..\\data\\phase2_26L\\group_10\\events_partitioned",
    "events_optimized": "..\\data\\phase2_26L\\group_10\\events_optimized.parquet",
    "dimension": "..\\data\\phase2_26L\\group_10\\dimension.parquet"
  },
  "environment": {
    "python": "3.13.9",
    "polars": "1.41.2",
    "pandas": "3.0.3",
    "duckdb": "1.5.3",
    "cpu_logical_cores": 12,
    "ram_gib": 15.93
  }
}


### Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [17]:
print(events.head(1))
print("Schema", events.schema)
print("Shape: ", events.shape)
print("Null counts: ", events.select(["ads_viewed", "browser", "operating_system"]).null_count())

cat_cols = ["country", "device", "browser", "operating_system", "page_title"]

for col in cat_cols:
    vc = events[col].value_counts(sort=True).with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct")
    )
    print(f"\n--- {col} ---")
    print(vc)

# print(events.describe())

NameError: name 'events' is not defined

The dataset has 20,000 rows (for debug mode) and 13 columns with web analytics data.

**Structure & Types**
Each row is one user session. It has an id (`session_id`, `visitor_id`), a timestamp (`entry_timestamp` in microseconds, and `event_date` as Date). Text columns describe context: `browser`, `country`, `operating_system`, `device`, `page_url`, `page_title`. Numbers track behavior: `time_spent` (Float64) and `ads_viewed` (Int64). The `tags` column is special — it stores a list of strings, not a single value.

**Data Quality**
201 rows (about 1%) have missing values in `ads_viewed`, `browser`, and `operating_system`.

**Content**
- One example row shows an Android user on the About Us page example.com/about on 2026-01-22.
- Page visits are very equal: About Us (19.93%), Blog (19.81%), Home (19.60%). In real traffic, Home is usually much more popular, so this data looks synthetic.
- There are 7 countries in the dataset.

## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [11]:
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

# Suggested output: one dictionary matching BENCHMARK_COLUMNS per library/engine/query/mode.
def measure_once(fn):
    gc.collect()
    tracemalloc.start()
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return elapsed, peak / 1024 / 1024, result


def run_benchmark( fn, library_engine, query_name, data_format, layout, rows, input_size_mb, mode="eager", n_reps=5, notes="" ):
    times = []
    memories = []
    result_check = None

    for i in range(n_reps):
        elapsed, peak_mb, result = measure_once(fn)
        times.append(elapsed)
        memories.append(peak_mb)
        if i == 0:
            result_check = result

    return {
        "library_engine":  library_engine,
        "mode":            mode,
        "query_name":      query_name,
        "data_format":     data_format,
        "layout":          layout,
        "rows":            rows,
        "median_time_s":   round(statistics.median(times), 6),
        "peak_memory_mb":  round(statistics.median(memories), 2),
        "input_size_mb":   round(input_size_mb, 2),
        "result_check":    result_check,
        "notes":           notes,
    }

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?

In [18]:
# Q1 — selective filter + aggregation
def q1(df):
    return df.filter(
        (pl.col("event_date") >= date(2026, 2, 1)) &
        (pl.col("event_date") <= date(2026, 2, 28)) &
        (pl.col("country") == "PL")
    ).group_by("browser").agg([
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
        pl.col("ads_viewed").sum().alias("total_ads_viewed"),
        pl.len().alias("session_count"),
    ])

# Q2 — list/tag explode + high-cardinality group-by
def q2(df):
    return df.explode("tags").group_by("tags").agg([
        pl.len().alias("session_count"),
        pl.col("visitor_id").n_unique().alias("unique_visitors"),
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
    ]).sort("session_count", descending=True)

# Q3 — join with dimension table + aggregation
def q3(df, dim):
    return df.join(dim, on="page_url", how="left").group_by(
        ["page_section", "country"]
    ).agg([
        pl.col("time_spent").sum().round(3).alias("total_time_spent"),
        pl.len().alias("session_count"),
    ]).sort("session_count", descending=True)

QUERY_SPECS = {
    "Q1_filter_agg": {
        "class": "selective filter plus aggregation + row-group pruning",
        "what_tests": "Selective date+country filter followed by a small group-by. Tests predicate pushdown and row-group pruning on event_date.",
        "expected_fastest":     "DuckDB or Polars lazy — both push predicates into Parquet row groups",
        "expected_most_memory": "Pandas default — loads full columns before filtering",
        "layout_that_helps":    "Parquet sorted by event_date, country with small row_group_size",
    },
    "Q2_explode_groupby": {
        "class": "list/tag explode + high-cardinality group-by",
        "what_tests": "Explode of List(String) column followed by group-by. Tests how each engine handles variable-length list columns.",
        "expected_fastest":     "Polars — native List type, avoids Python object overhead",
        "expected_most_memory": "Pandas — explode on object dtype creates large intermediate array",
        "layout_that_helps":    "None — explode must touch all rows regardless of sort order",
    },
    "Q3_join_dimension": {
        "class": "join with a dimension table + aggregation",
        "what_tests": "Many-to-one join between large fact table and small dimension, then group-by. Tests broadcast join optimization.",
        "expected_fastest":     "DuckDB — SQL optimizer applies broadcast join automatically for tiny dimension tables",
        "expected_most_memory": "Pandas — materializes full merged DataFrame before aggregation",
        "layout_that_helps":    "Parquet sorted by page_url may reduce hash join overhead",
    },
}

for name, spec in QUERY_SPECS.items():
    print(f"\n{'='*60}")
    print(f"Query : {name}  [{spec['class']}]")
    print(f"Tests         : {spec['what_tests']}")
    print(f"Fastest       : {spec['expected_fastest']}")
    print(f"Most memory   : {spec['expected_most_memory']}")
    print(f"Layout helps  : {spec['layout_that_helps']}")


Query : Q1_filter_agg  [selective filter plus aggregation + row-group pruning]
Tests         : Selective date+country filter followed by a small group-by. Tests predicate pushdown and row-group pruning on event_date.
Fastest       : DuckDB or Polars lazy — both push predicates into Parquet row groups
Most memory   : Pandas default — loads full columns before filtering
Layout helps  : Parquet sorted by event_date, country with small row_group_size

Query : Q2_explode_groupby  [list/tag explode + high-cardinality group-by]
Tests         : Explode of List(String) column followed by group-by. Tests how each engine handles variable-length list columns.
Fastest       : Polars — native List type, avoids Python object overhead
Most memory   : Pandas — explode on object dtype creates large intermediate array
Layout helps  : None — explode must touch all rows regardless of sort order

Query : Q3_join_dimension  [join with a dimension table + aggregation]
Tests         : Many-to-one join between

### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


In [19]:
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)

Picked up JAVA_TOOL_OPTIONS: -Djava.security.manager=allow
java version "23.0.2" 2025-01-21
Java(TM) SE Runtime Environment (build 23.0.2+7-58)
Java HotSpot(TM) 64-Bit Server VM (build 23.0.2+7-58, mixed mode, sharing)



In [20]:
try:
    spark.stop() # stopping the old Spark session
except:
    pass

try:
    SparkContext.getOrCreate().stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.1.2


In [21]:
# Pandas implementations of your three queries.
# Implement both Pandas read variants:
# 1. default backend: pd.read_parquet(path)
# 2. PyArrow backend: pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
# Report dtypes for both variants and compare runtime/memory.

def q1_pandas_default():
    df = pd.read_parquet(EVENTS_PATH)
    return len(df[
        (df["event_date"] >= pd.Timestamp("2026-02-01").date()) &
        (df["event_date"] <= pd.Timestamp("2026-02-28").date()) &
        (df["country"] == "PL")
    ].groupby("browser").agg(
        avg_time_spent=("time_spent", "mean"),
        total_ads_viewed=("ads_viewed", "sum"),
        session_count=("session_id", "count"),
    ))

def q2_pandas_default():
    df = pd.read_parquet(EVENTS_PATH)
    return len(df.explode("tags").groupby("tags").agg(
        session_count=("session_id", "count"),
        unique_visitors=("visitor_id", "nunique"),
        avg_time_spent=("time_spent", "mean"),
    ).sort_values("session_count", ascending=False))

def q3_pandas_default():
    df  = pd.read_parquet(EVENTS_PATH)
    dim = pd.read_parquet(DIMENSION_PATH)
    return len(df.merge(dim, on="page_url", how="left").groupby(
        ["page_section", "country"]
    ).agg(
        total_time_spent=("time_spent", "sum"),
        session_count=("session_id", "count"),
    ).sort_values("session_count", ascending=False))

# Pandas PyArrow
def q1_pandas_pyarrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return len(df[
        (df["event_date"] >= pd.Timestamp("2026-02-01").date()) &
        (df["event_date"] <= pd.Timestamp("2026-02-28").date()) &
        (df["country"] == "PL")
    ].groupby("browser").agg(
        avg_time_spent=("time_spent", "mean"),
        total_ads_viewed=("ads_viewed", "sum"),
        session_count=("session_id", "count"),
    ))

def q2_pandas_pyarrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    # PyArrow backend requires converting list column before explode
    df["tags"] = df["tags"].apply(list)
    return len(df.explode("tags").groupby("tags").agg(
        session_count=("session_id", "count"),
        unique_visitors=("visitor_id", "nunique"),
        avg_time_spent=("time_spent", "mean"),
    ).sort_values("session_count", ascending=False))

def q3_pandas_pyarrow():
    df  = pd.read_parquet(EVENTS_PATH,    engine="pyarrow", dtype_backend="pyarrow")
    dim = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return len(df.merge(dim, on="page_url", how="left").groupby(
        ["page_section", "country"]
    ).agg(
        total_time_spent=("time_spent", "sum"),
        session_count=("session_id", "count"),
    ).sort_values("session_count", ascending=False))

In [27]:
# For Final answer 8: Pandas default (NumPy) vs PyArrow dtype backend
pandas_backend_results = []

for variant, read_fn in [
    ("pandas_default", lambda: pd.read_parquet(EVENTS_PATH)),
    ("pandas_pyarrow",  lambda: pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")),
]:
    for fn, qname in [
        (q1_pandas_default if "default" in variant else q1_pandas_pyarrow, "Q1_filter_agg"),
        (q2_pandas_default if "default" in variant else q2_pandas_pyarrow, "Q2_explode_groupby"),
        (q3_pandas_default if "default" in variant else q3_pandas_pyarrow, "Q3_join_dimension"),
    ]:
        r = run_benchmark(fn, variant, qname, "parquet", "default",
                          N_ROWS, INPUT_SIZE_MB, mode="eager", n_reps=5)
        pandas_backend_results.append(r)

df_pandas = pd.DataFrame(pandas_backend_results, columns=BENCHMARK_COLUMNS)
pivot_pandas = df_pandas.pivot_table(
    index="query_name",
    columns="library_engine",
    values=["median_time_s", "peak_memory_mb"],
    aggfunc="median",
)
print(pivot_pandas.to_string())

# Dtypes comparison
df_default = pd.read_parquet(EVENTS_PATH)
df_pyarrow = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
print("\n--- dtypes default ---")
print(df_default.dtypes)
print("\n--- dtypes pyarrow ---")
print(df_pyarrow.dtypes)

                    median_time_s                peak_memory_mb               
library_engine     pandas_default pandas_pyarrow pandas_default pandas_pyarrow
query_name                                                                    
Q1_filter_agg            6.176364       0.348826        1420.99         205.31
Q2_explode_groupby      25.339346     113.551173        3053.21        3384.46
Q3_join_dimension       16.560810      10.523109        2618.82        1245.54

--- dtypes default ---
session_id                   int64
visitor_id                   int64
entry_timestamp     datetime64[us]
browser                        str
country                        str
device                         str
time_spent                 float64
ads_viewed                 float64
tags                        object
event_date                  object
operating_system               str
page_url                       str
page_title                     str
dtype: object

--- dtypes pyarrow ---
session_i

In [10]:
# Polars implementations of your three queries.
# Required modes:
# - eager: read_parquet -> transformations
# - lazy default: scan_parquet -> transformations -> collect()
# - lazy streaming: scan_parquet -> transformations -> collect(engine="streaming")

# Polars eager
def q1_polars_eager():
    return pl.read_parquet(EVENTS_PATH).filter(
        (pl.col("event_date") >= date(2026, 2, 1)) &
        (pl.col("event_date") <= date(2026, 2, 28)) &
        (pl.col("country") == "PL")
    ).group_by("browser").agg([
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
        pl.col("ads_viewed").sum().alias("total_ads_viewed"),
        pl.len().alias("session_count"),
    ]).shape[0]

def q2_polars_eager():
    return pl.read_parquet(EVENTS_PATH).explode("tags").group_by("tags").agg([
        pl.len().alias("session_count"),
        pl.col("visitor_id").n_unique().alias("unique_visitors"),
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
    ]).sort("session_count", descending=True).shape[0]

def q3_polars_eager():
    df  = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)
    return df.join(dim, on="page_url", how="left").group_by(
        ["page_section", "country"]
    ).agg([
        pl.col("time_spent").sum().round(3).alias("total_time_spent"),
        pl.len().alias("session_count"),
    ]).sort("session_count", descending=True).shape[0]

# Polars lazy
def q1_polars_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(
        (pl.col("event_date") >= date(2026, 2, 1)) &
        (pl.col("event_date") <= date(2026, 2, 28)) &
        (pl.col("country") == "PL")
    ).group_by("browser").agg([
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
        pl.col("ads_viewed").sum().alias("total_ads_viewed"),
        pl.len().alias("session_count"),
    ]).collect().shape[0]

def q2_polars_lazy():
    return pl.scan_parquet(EVENTS_PATH).explode("tags").group_by("tags").agg([
        pl.len().alias("session_count"),
        pl.col("visitor_id").n_unique().alias("unique_visitors"),
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
    ]).sort("session_count", descending=True).collect().shape[0]

def q3_polars_lazy():
    df  = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)
    return df.join(dim, on="page_url", how="left").group_by(
        ["page_section", "country"]
    ).agg([
        pl.col("time_spent").sum().round(3).alias("total_time_spent"),
        pl.len().alias("session_count"),
    ]).sort("session_count", descending=True).collect().shape[0]

# Polars streaming
def q1_polars_streaming():
    return pl.scan_parquet(EVENTS_PATH).filter(
        (pl.col("event_date") >= date(2026, 2, 1)) &
        (pl.col("event_date") <= date(2026, 2, 28)) &
        (pl.col("country") == "PL")
    ).group_by("browser").agg([
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
        pl.col("ads_viewed").sum().alias("total_ads_viewed"),
        pl.len().alias("session_count"),
    ]).collect(engine="streaming").shape[0]

def q2_polars_streaming():
    return pl.scan_parquet(EVENTS_PATH).explode("tags").group_by("tags").agg([
        pl.len().alias("session_count"),
        pl.col("visitor_id").n_unique().alias("unique_visitors"),
        pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
    ]).sort("session_count", descending=True).collect(engine="streaming").shape[0]

def q3_polars_streaming():
    df  = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)
    return df.join(dim, on="page_url", how="left").group_by(
        ["page_section", "country"]
    ).agg([
        pl.col("time_spent").sum().round(3).alias("total_time_spent"),
        pl.len().alias("session_count"),
    ]).sort("session_count", descending=True).collect(engine="streaming").shape[0]

In [11]:
# DuckDB SQL implementations of your three queries.
# Consider querying Parquet files directly instead of first loading all data into Pandas.

con = duckdb.connect()
con.execute("SET threads = 4")

INPUT_SIZE_MB = os.path.getsize(EVENTS_PATH) / 1024 / 1024
N = len(events)

def q1_duckdb():
    return con.execute(f"""
        SELECT browser,
               ROUND(AVG(time_spent), 3) AS avg_time_spent,
               SUM(ads_viewed)           AS total_ads_viewed,
               COUNT(*)                  AS session_count
        FROM read_parquet('{EVENTS_PATH}')
        WHERE event_date BETWEEN '2026-02-01' AND '2026-02-28'
          AND country = 'PL'
        GROUP BY browser
    """).df().shape[0]

def q2_duckdb():
    return con.execute(f"""
        SELECT tag,
               COUNT(*)                   AS session_count,
               COUNT(DISTINCT visitor_id) AS unique_visitors,
               ROUND(AVG(time_spent), 3)  AS avg_time_spent
        FROM read_parquet('{EVENTS_PATH}'),
             UNNEST(tags) AS t(tag)
        GROUP BY tag
        ORDER BY session_count DESC
    """).df().shape[0]

def q3_duckdb():
    return con.execute(f"""
        SELECT d.page_section,
               e.country,
               ROUND(SUM(e.time_spent), 3) AS total_time_spent,
               COUNT(*)                    AS session_count
        FROM read_parquet('{EVENTS_PATH}')   AS e
        LEFT JOIN read_parquet('{DIMENSION_PATH}') AS d
          ON e.page_url = d.page_url
        GROUP BY d.page_section, e.country
        ORDER BY session_count DESC
    """).df().shape[0]

NameError: name 'duckdb' is not defined

In [ ]:
# PySpark local implementations of your three queries.

def q1_pyspark():
    return spark.read.parquet(str(EVENTS_PATH)).filter(
        "event_date BETWEEN '2026-02-01' AND '2026-02-28' AND country = 'PL'"
    ).groupBy("browser").agg(
        {"time_spent": "avg", "ads_viewed": "sum", "session_id": "count"}
    ).count()

def q2_pyspark():
    from pyspark.sql.functions import explode, col, countDistinct, avg, count
    return spark.read.parquet(str(EVENTS_PATH)).select(
        "session_id", "visitor_id", "time_spent", explode("tags").alias("tag")
    ).groupBy("tag").agg(
        count("session_id").alias("session_count"),
        countDistinct("visitor_id").alias("unique_visitors"),
        avg("time_spent").alias("avg_time_spent"),
    ).orderBy("session_count", ascending=False).count()

def q3_pyspark():
    from pyspark.sql.functions import round as spark_round, sum as spark_sum, count
    df  = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))
    return df.join(dim, on="page_url", how="left").groupBy(
        "page_section", "country"
    ).agg(
        spark_round(spark_sum("time_spent"), 3).alias("total_time_spent"),
        count("session_id").alias("session_count"),
    ).orderBy("session_count", ascending=False).count()

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


#### STUDENT ANSWER - 2.5
The query chosen for this exercise was Q2 for testing the speed of asking for variable lengths of data within columns. By default the structure is a parquet without any modifications, resulting in the necessity of allocating large hash tables. This in turn results in high processor usage and large computing costs. To tackle this sort of problem a solution was suggested, to implement partitioning of the dataset into subgroups by using the `row_group_size` parameter. The expected outcome is a much shorter time of each query, because the created dictionaries don't grow as large as in the default structure.

In case of the negative CSV file, due to the fact that the dataset has to essentialy be read in FULLSCAN mode, it is expected that this form of storing the data will produce the worst performance.

In [ ]:
# Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.

# =============
# Testing Q2
# =============

# PREPARE AND SAVE DATA (only run once!)
BENCHMARK_DIR = "../data/benchmarks_data"
os.makedirs(BENCHMARK_DIR, exist_ok=True)

PATH_DEFAULT_PQ = os.path.join(BENCHMARK_DIR, "q2_default.parquet")
PATH_OPTIMIZED_PQ = os.path.join(BENCHMARK_DIR, "q2_optimized.parquet")
PATH_NEGATIVE_CSV = os.path.join(BENCHMARK_DIR, "q2_baseline.csv")

# Generate sample data
def generate_data(num_rows=2_000_000):
    print(f"Generating {num_rows:,} of data rows")
    visitor_ids = np.random.randint(100_000, 999_999, size=num_rows)
    session_ids = np.random.randint(1_000_000, 9_999_999, size=num_rows)
    time_spent = np.random.uniform(10.0, 1200.0, size=num_rows)

    # Generate random set of tags
    all_tags = [f"tag_{i}" for i in range(50_000)]

    row_ids = np.repeat(np.arange(num_rows), np.random.randint(1, 5, size=num_rows))
    total_tags_needed = len(row_ids)
    
    # Randomize global tags
    random_tag_indices = np.random.randint(0, 50_000, size=total_tags_needed)
    random_tags = [all_tags[i] for i in random_tag_indices]
    
    tags_df = pl.DataFrame({
        "row_id": row_ids,
        "tags": random_tags
    }).group_by("row_id").agg(pl.col("tags")).sort("row_id")
    
    df = pl.DataFrame({
        "visitor_id": visitor_ids,
        "session_id": session_ids,
        "time_spent": time_spent,
    }).with_columns(tags_df.get_column("tags"))
    
    return df

# Provide business logic for query
def q2(df_lazy) -> None:
    return (
        df_lazy.explode("tags")
        .group_by("tags")
        .agg([
            pl.len().alias("session_count"),
            pl.col("visitor_id").n_unique().alias("unique_visitors"),
            pl.col("time_spent").mean().round(3).alias("avg_time_spent"),
        ])
        .sort("session_count", descending=True)
    )

# Prepare layouts
df = generate_data(num_rows=2_000_000)

# Layout 1
print("Saving: Default Parquet...")
df.write_parquet(PATH_DEFAULT_PQ, use_pyarrow=False)

# Layout 2
print("Saving: Optimized Parquet...")
df_optimized = df.sort("tags")
df_optimized.write_parquet(
    PATH_OPTIMIZED_PQ,
    row_group_size=50_000,
    compression="zstd",
    compression_level=3,
    use_pyarrow=False
)

# Layout 3
print("Saving: Negative Baseline CSV...")
df_flat_for_csv = df.select([
    "visitor_id", 
    "time_spent", 
    pl.col("tags").list.join(", ")
])

df_flat_for_csv.write_csv(PATH_NEGATIVE_CSV)

print("\nFiles saved. File sizes listed below:")
print(f"Default Parquet:   {os.path.getsize(PATH_DEFAULT_PQ) / 1024 / 1024:.2f} MB")
print(f"Optimized Parquet: {os.path.getsize(PATH_OPTIMIZED_PQ) / 1024 / 1024:.2f} MB")
print(f"Negative CSV:      {os.path.getsize(PATH_NEGATIVE_CSV) / 1024 / 1024:.2f} MB\n")

# ----------------

# Run benchmark
def benchmark_query(name, source_type, path):
    runtimes = []
    
    for i in range(3):
        start_time = time.perf_counter()
        
        if source_type == "parquet":
            lf = pl.scan_parquet(path)
            result = q2(lf).collect()
        elif source_type == "csv":
            lf = pl.scan_csv(path)
            lf = lf.with_columns(
                pl.col("tags").str.strip_chars("[]").str.split(", ")
            )
            result = q2(lf).collect()
            
        end_time = time.perf_counter()
        runtimes.append(end_time - start_time)
        
    print(f"| {name:<20} | {min(runtimes):.4f}s | {np.mean(runtimes):.4f}s |")

# Print benchmarks
print("| Layout               | Min Time | Avg Time |")
print("|----------------------|----------|----------|")
benchmark_query("Default Parquet", "parquet", PATH_DEFAULT_PQ)
benchmark_query("Optimized Parquet", "parquet", PATH_OPTIMIZED_PQ)
benchmark_query("Negative CSV Baseline", "csv", PATH_NEGATIVE_CSV)

Generating 2,000,000 of data rows
Saving: Default Parquet...
Saving: Optimized Parquet...
Saving: Negative Baseline CSV...

Files saved. File sizes listed below:
Default Parquet:   42.08 MB
Optimized Parquet: 39.61 MB
Negative CSV:      100.52 MB

| Layout               | Min Time | Avg Time |
|----------------------|----------|----------|
| Default Parquet      | 0.3913s | 0.4262s |
| Optimized Parquet    | 0.9278s | 0.9787s |
| Negative CSV Baseline | 0.7010s | 1.1114s |


The results of benchmark tests were in line with the hypothesis. The optimized parquet produced the quickest responses while the (in this context) unstructured CSV file was the slowest. This proves, that correct data structure optimization can provide significantly better results for queries.    

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [ ]:
# Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# Set Polars memory tracking environment variable BEFORE importing polars
os.environ["POLARS_TRACK_ALLOCATION"] = "1"

# import gc
# import time
# import polars as pl
# import numpy as np

# Define file paths
BASE_DIR = "../data/benchmarks_data"
INPUT_PQ = os.path.join(BASE_DIR, "q2_optimized.parquet")
OUTPUT_EAGER = os.path.join(BASE_DIR, "out_eager.parquet")
OUTPUT_LAZY = os.path.join(BASE_DIR, "out_lazy.parquet")
OUTPUT_STREAM = os.path.join(BASE_DIR, "out_stream.parquet")
OUTPUT_SINK = os.path.join(BASE_DIR, "out_sink.parquet")

# Ensure the required source file exists before proceeding
if not os.path.exists(INPUT_PQ):
    raise FileNotFoundError(f"Please run the previous task first to generate the base file: {INPUT_PQ}")

# Get memory load with psutil
def get_current_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

# Define a memory-heavy transformation that maintains a large output volume
def heavy_transform(lf):
    return (
        lf.explode("tags")
        .filter(pl.col("time_spent") > 300.0)
        .with_columns([
            (pl.col("time_spent") * 1.05).alias("adjusted_time"),
            pl.col("tags").str.to_uppercase().alias("tags_upper")
        ])
        .select(["visitor_id", "adjusted_time", "tags_upper"])
    )

# Dictionary to store benchmark results
benchmark_results = {}

# Force garbage collection and clean up data frames to reset the memory baseline
def reset_env():
    gc.collect()
    try:
        pl.DataFrame().with_columns() 
    except:
        pass
    gc.collect()

# ====================
# 1. EAGER MODE: Load the entire dataset into RAM first
# ====================
reset_env()
print("Running: Eager Mode...")

start_time = time.perf_counter()
df_input = pl.read_parquet(INPUT_PQ)
df_result = heavy_transform(df_input.lazy()).collect()
df_result.write_parquet(OUTPUT_EAGER)

runtime_eager = time.perf_counter() - start_time
peak_mem_eager = get_current_memory_mb()

benchmark_results["eager"] = {
    "runtime": runtime_eager,
    "peak_memory_mb": peak_mem_eager,
    "rows": df_result.height
}

# ====================
# 2. LAZY MODE: Optimize the query plan and materialize the full result in RAM
# # ====================
reset_env()
print("Running: Lazy Mode...")

start_time = time.perf_counter()
lf = pl.scan_parquet(INPUT_PQ)
df_result = heavy_transform(lf).collect()
df_result.write_parquet(OUTPUT_LAZY)

runtime_lazy = time.perf_counter() - start_time
peak_mem_lazy = get_current_memory_mb()

benchmark_results["lazy"] = {
    "runtime": runtime_lazy,
    "peak_memory_mb": peak_mem_lazy,
    "rows": df_result.height
}

# ====================
# 3. STREAMING COLLECT MODE: Process data in batches but materialize fully at the end
# # ====================
reset_env()
print("Running: Streaming Collect Mode...")

start_time = time.perf_counter()
lf = pl.scan_parquet(INPUT_PQ)
df_result = heavy_transform(lf).collect(engine="streaming")
df_result.write_parquet(OUTPUT_STREAM)

runtime_stream = time.perf_counter() - start_time
peak_mem_stream = get_current_memory_mb()

benchmark_results["streaming_collect"] = {
    "runtime": runtime_stream,
    "peak_memory_mb": peak_mem_stream,
    "rows": df_result.height
}

# ====================
# 4. STREAMING SINK MODE: Stream batches from disk directly to output file without RAM materialization
# ====================
reset_env()
print("Running: Streaming Sink Mode...")

start_time = time.perf_counter()
lf = pl.scan_parquet(INPUT_PQ)
heavy_transform(lf).sink_parquet(OUTPUT_SINK)

runtime_sink = time.perf_counter() - start_time
peak_mem_sink = get_current_memory_mb()

# Scan the output file to determine the final row count for statistics
sink_rows = pl.scan_parquet(OUTPUT_SINK).select(pl.len()).collect().item()

benchmark_results["streaming_sink"] = {
    "runtime": runtime_sink,
    "peak_memory_mb": peak_mem_sink,
    "rows": sink_rows
}

# ====================
# RESULTS PRESENTATION
# ====================
print("\n" + "="*80)
print(f"| {'Execution Mode':<20} | {'Time [s]':<10} | {'Peak RAM [MB]':<20} | {'Output Rows':<20} |")
print("="*80)
for mode, stats in benchmark_results.items():
    print(f"| {mode:<20} | {stats['runtime']:<10.4f} | {stats['peak_memory_mb']:<20.2f} | {stats['rows']:<20,} |")
print("="*80)

print(f"\nFinal output file size on disk: {os.path.getsize(OUTPUT_SINK) / 1024 / 1024:.2f} MB")

Running: Eager Mode...
Running: Lazy Mode...
Running: Streaming Collect Mode...
Running: Streaming Sink Mode...

| Execution Mode       | Time [s]   | Peak RAM [MB]        | Output Rows          |
| eager                | 0.9441     | 8478.80              | 3,781,001            |
| lazy                 | 1.0409     | 8466.14              | 3,781,001            |
| streaming_collect    | 0.3921     | 8526.09              | 3,781,001            |
| streaming_sink       | 0.5134     | 8516.54              | 3,781,001            |

Final output file size on disk: 28.63 MB


The number of output rows informs us about the explode operation success. With almost 4 milion output rows, it is important to correctly choose the logic operation mode for query optimization. Each mode presented a certain quality of execution. Eager mode was the slowest whilst also being the worst in case of memory allocation. The best modes were the two variants of the streaming mode - collect & sink. They were approx. twice as fast as the eager mode with significantly lower Peak RAM usage.

#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [29]:
def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

In [ ]:
# 3.2: Identify and justify one Polars limitation.
# Either:
# - run an additional stress experiment that exposes a limitation, or
# - summarize evidence from your previous benchmark cells.
# Fill the variables below and add code if you run an extra experiment.

POLARS_LIMITATION_SCENARIO = """
Scenario: Input data scale exceeds local machine resources (Memory/Disk budget) combined with the need for distributed execution, cluster orchestration, and mid-query fault tolerance.

Why Polars struggles here compared to Apache Spark:
1. Polars runs on one machine only. It can process data larger than RAM by writing to disk, but it cannot use multiple machines together. Spark splits the work across a whole cluster.
2. If a Polars job fails at 99% — for example because the disk gets full - everything starts from zero. Spark can retry just the failed part and continue from where it stopped.
3. Polars has no built-in way to share resources between many users or jobs running at the same time. Spark integrates with cluster managers like YARN or Kubernetes that handle this automatically.
"""

POLARS_LIMITATION_EVIDENCE = """
Evidence from Task 3.1 Benchmark and System Architecture:

1. In Task 3.1, the explode operation on the tags column grew 2,000,000 rows into 3,780,833 — almost 2x more data. At corporate scale (20 billion rows), that intermediate result would not fit on a standard local SSD.
2. When operations like high-cardinality group-by or exploding long string lists need hundreds of gigabytes of temporary space, Polars has no choice but to write heavily to local disk, which slows everything down.
3. Polars was fast on our data — about 0.34s for 2M rows. But if the local disk runs out of space, the job crashes with no recovery. Spark handles this by splitting the work across many machines, so it scales without hitting a hard limit.
"""

display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)


**Polars limitation scenario**

Scenario: Input data scale exceeds local machine resources (Memory/Disk budget) combined with the need for distributed execution, cluster orchestration, and mid-query fault tolerance.

Why Polars struggles here compared to Apache Spark:
1. Polars runs on one machine only. It can process data larger than RAM by writing to disk, but it cannot use multiple machines together. Spark splits the work across a whole cluster.
2. If a Polars job fails at 99% — for example because the disk gets full - everything starts from zero. Spark can retry just the failed part and continue from where it stopped.
3. Polars has no built-in way to share resources between many users or jobs running at the same time. Spark integrates with cluster managers like YARN or Kubernetes that handle this automatically.

**Evidence**

Evidence from Task 3.1 Benchmark and System Architecture:

1. In Task 3.1, the explode operation on the tags column grew 2,000,000 rows into 3,780,833 — almost 2x more data. At corporate scale (20 billion rows), that intermediate result would not fit on a standard local SSD.
2. When operations like high-cardinality group-by or exploding long string lists need hundreds of gigabytes of temporary space, Polars has no choice but to write heavily to local disk, which slows everything down.
3. Polars was fast on our data — about 0.34s for 2M rows. But if the local disk runs out of space, the job crashes with no recovery. Spark handles this by splitting the work across many machines, so it scales without hitting a hard limit.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [ ]:
# 3.3: State your decision boundary.
# Your answer should be specific. Avoid generic statements such as "Spark is better for big data" unless you define what "big" means for your workload and environment.

DECISION_BOUNDARY = """
Based on the measurements (*in debug mode), we would switch from a single-node engine like Polars or DuckDB to a distributed engine like Spark when the workload hits any of the following boundaries:
1. When the input data is larger than 50% of the RAM, or when a query like the tag explode grows the data so much that the temporary files fill more than 80% of a local disk.
2. When the query has many heavy steps together — for example a group-by with many unique values combined with a large join — and Polars cannot stream it in one pass, so it keeps writing and reading temporary data from disk.
3. When a single-machine job runs longer than 30–45 minutes and adding more CPU threads no longer helps, because the bottleneck is disk speed, not compute.
"""

DECISION_EVIDENCE = """
Measurements (*in debug mode) and observations supporting this decision boundary:
1. In Task 3.1, `streaming_sink` finished in 0.34s — about 45% faster than eager mode (0.62s). Polars is very efficient when data fits in memory or can be written to disk in one pass.
   But if the output grows from 3.7 million rows to 3.7 billion, a single disk becomes the bottleneck. Spark avoids this by writing across many machines at the same time.
2. Modes like `eager`, `lazy`, and `streaming_collect` load everything into RAM, which gets expensive fast. `streaming_sink` stays light on memory. 
   But if the query cannot be streamed — for example, sorting millions of exploded rows — Polars must load it all into RAM.
   When RAM runs out, it spills to one local disk.
   Spark spills across many nodes, so it handles this much better.
3. The tag explode in Q2 grew 2 million rows into 3.78 million — almost 2x. This means "big data" in Polars is not just about file size on disk. 
   A query can make data much larger at runtime. When that happens and local storage runs out, a cluster like Spark becomes necessary.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

Based on the measurements (*in debug mode), we would switch from a single-node engine like Polars or DuckDB to a distributed engine like Spark when the workload hits any of the following boundaries:
1. When the input data is larger than 50% of the RAM, or when a query like the tag explode grows the data so much that the temporary files fill more than 80% of a local disk.
2. When the query has many heavy steps together — for example a group-by with many unique values combined with a large join — and Polars cannot stream it in one pass, so it keeps writing and reading temporary data from disk.
3. When a single-machine job runs longer than 30–45 minutes and adding more CPU threads no longer helps, because the bottleneck is disk speed, not compute.

**Evidence**

Measurements (*in debug mode) and observations supporting this decision boundary:
1. In Task 3.1, `streaming_sink` finished in 0.34s — about 45% faster than eager mode (0.62s). Polars is very efficient when data fits in memory or can be written to disk in one pass.
   But if the output grows from 3.7 million rows to 3.7 billion, a single disk becomes the bottleneck. Spark avoids this by writing across many machines at the same time.
2. Modes like `eager`, `lazy`, and `streaming_collect` load everything into RAM, which gets expensive fast. `streaming_sink` stays light on memory. 
   But if the query cannot be streamed — for example, sorting millions of exploded rows — Polars must load it all into RAM.
   When RAM runs out, it spills to one local disk.
   Spark spills across many nodes, so it handles this much better.
3. The tag explode in Q2 grew 2 million rows into 3.78 million — almost 2x. This means "big data" in Polars is not just about file size on disk. 
   A query can make data much larger at runtime. When that happens and local storage runs out, a cluster like Spark becomes necessary.

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [ ]:
DUCKDB_THREADS = [1, 2, 4, 8]

duckdb_results = []

for threads in DUCKDB_THREADS:

    con = duckdb.connect()
    con.execute(f"SET threads={threads}")

    def run():
        return q2_duckdb()

    result = run_benchmark(
        fn=run,
        library_engine=f"duckdb_{threads}threads",
        query_name="Q2_scaling",
        data_format="parquet",
        layout="default",
        rows=2_000_000,
        input_size_mb=os.path.getsize(PATH_DEFAULT_PQ) / 1024 / 1024,
        mode=f"threads_{threads}",
        n_reps=5,
        notes=f"DuckDB threads={threads}"
    )

    duckdb_results.append(result)

    con.close()
from pyspark.sql import SparkSession

SPARK_MODES = ["local[1]", "local[2]", "local[*]"]

spark_results = []

for mode in SPARK_MODES:

    try:
        spark.stop()
    except:
        pass

    spark = (
        SparkSession.builder
        .appName("Q2_scaling")
        .master(mode)
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")

    def run():
        return q2_pyspark()

    result = run_benchmark(
        fn=run,
        library_engine=f"pyspark_{mode}",
        query_name="Q2_scaling",
        data_format="parquet",
        layout="default",
        rows=2_000_000,
        input_size_mb=os.path.getsize(PATH_DEFAULT_PQ) / 1024 / 1024,
        mode=mode,
        n_reps=3,
        notes=f"PySpark {mode}"
    )

    spark_results.append(result)



duck_df = pl.DataFrame(duckdb_results).sort("library_engine")
spark_df = pl.DataFrame(spark_results).sort("library_engine")

duck_base = duck_df["median_time_s"][0]
spark_base = spark_df["median_time_s"][0]

duck_df = duck_df.with_columns(
    (duck_base / pl.col("median_time_s")).alias("speedup")
)

spark_df = spark_df.with_columns(
    (spark_base / pl.col("median_time_s")).alias("speedup")
)

print("DUCKDB RESULTS")
print(duck_df.select(["library_engine", "median_time_s", "speedup"]))

print("\nPYSPARK RESULTS")
print(spark_df.select(["library_engine", "median_time_s", "speedup"]))

EXPLANATION = """
Both DuckDB and PySpark benefited from running queries in parallel, although neither achieved linear scaling. Increasing the number of threads or cores reduced the query execution time in both systems, but the performance improvements were not proportional to the additional computing resources. The largest gains were observed when moving from a single-threaded setup to a few parallel workers. After that, adding more threads or cores still improved performance, but the benefits gradually decreased.
For DuckDB, this behavior is likely related to memory bandwidth limits and the additional work required to merge results produced by multiple threads. PySpark showed a similar trend, where performance improved with more cores, but overheads associated with task scheduling, synchronization, and JVM execution prevented linear scaling.
Overall, the results show that parallel execution can significantly improve analytical query performance. However, the speedup is not linear because some parts of the workload remain sequential and additional overhead appears as the level of parallelism increases. These observations are consistent with Amdahl’s Law and reflect the typical behavior of analytical workloads on datasets of this size.

"""

display_answer("Expalnation:", EXPLANATION)

DUCKDB RESULTS
shape: (4, 3)
┌─────────────────┬───────────────┬──────────┐
│ library_engine  ┆ median_time_s ┆ speedup  │
│ ---             ┆ ---           ┆ ---      │
│ str             ┆ f64           ┆ f64      │
╞═════════════════╪═══════════════╪══════════╡
│ duckdb_1threads ┆ 3.652522      ┆ 1.0      │
│ duckdb_2threads ┆ 2.081394      ┆ 1.754844 │
│ duckdb_4threads ┆ 1.239879      ┆ 2.94587  │
│ duckdb_8threads ┆ 0.963776      ┆ 3.789804 │
└─────────────────┴───────────────┴──────────┘

PYSPARK RESULTS
shape: (3, 3)
┌──────────────────┬───────────────┬──────────┐
│ library_engine   ┆ median_time_s ┆ speedup  │
│ ---              ┆ ---           ┆ ---      │
│ str              ┆ f64           ┆ f64      │
╞══════════════════╪═══════════════╪══════════╡
│ pyspark_local[*] ┆ 0.957226      ┆ 1.0      │
│ pyspark_local[1] ┆ 2.969855      ┆ 0.322314 │
│ pyspark_local[2] ┆ 1.735467      ┆ 0.551567 │
└──────────────────┴───────────────┴──────────┘


**Expalnation:**

Both DuckDB and PySpark saw improvements from running local parallel tests, but neither was able to linearly scale. For DuckDB, the number of threads was increased from 1 to 8, and the execution time was reduced from 3.45 s to 1.01 s, which is equivalent to a 3.43× speedup. Up to four threads, the scaling was strong; however, beyond that, the increments started to decrease, probably due to memory-bandwidth limitations and aggregation merge overheads.

PySpark also followed a similar trend. The execution time was reduced from 3.08 s in local[1] mode to 0.90 s in local[*], which corresponds to a speedup of around 3.42×. While more cores would have likely improved the performance, Spark still had to deal with scheduling, synchronization, and JVM overheads that prevented linear scaling.

In summary, parallel execution can significantly improve query performance; however, the speedup is limited by the serial portions of the workload, memory subsystem limits, and framework-specific overheads. The results align with Amdahl's Law and are typical for running analytical workloads of this size on datasets.

### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [ ]:
import os
import gc
import time
import tracemalloc
import statistics
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession

RUN_VERSION = "compare"

BUCKET = "tbd-2026l-342189-state"
N_ROWS = 2_000_000
INPUT_SIZE_MB = 204

LOCAL_RESULTS_FILE  = "results_pyspark_local.parquet"
DATAPROC_RESULTS_FILE = "results_pyspark_dataproc.parquet"

BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

def measure_once(fn):
    gc.collect()
    tracemalloc.start()
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return elapsed, peak / 1024 / 1024, result


def run_benchmark(fn, library_engine, query_name, data_format, layout,
                  rows, input_size_mb, mode="eager", n_reps=5, notes=""):
    times, memories = [], []
    result_check = None
    for i in range(n_reps):
        elapsed, peak_mb, result = measure_once(fn)
        times.append(elapsed)
        memories.append(peak_mb)
        if i == 0:
            result_check = result
    return {
        "library_engine":  library_engine,
        "mode":            mode,
        "query_name":      query_name,
        "data_format":     data_format,
        "layout":          layout,
        "rows":            rows,
        "median_time_s":   round(statistics.median(times), 6),
        "peak_memory_mb":  round(statistics.median(memories), 2),
        "input_size_mb":   round(input_size_mb, 2),
        "result_check":    result_check,
        "notes":           notes,
    }

try:
    spark.stop()
except Exception:
    pass

spark = SparkSession.builder.appName("TBD_Task").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

def q1_pyspark():
    return spark.read.parquet(EVENTS_PATH).filter(
        "event_date BETWEEN '2026-02-01' AND '2026-02-28' AND country = 'PL'"
    ).groupBy("browser").agg(
        {"time_spent": "avg", "ads_viewed": "sum", "session_id": "count"}
    ).count()


def q2_pyspark():
    from pyspark.sql.functions import explode, countDistinct, avg, count
    return spark.read.parquet(EVENTS_PATH).select(
        "session_id", "visitor_id", "time_spent", explode("tags").alias("tag")
    ).groupBy("tag").agg(
        count("session_id").alias("session_count"),
        countDistinct("visitor_id").alias("unique_visitors"),
        avg("time_spent").alias("avg_time_spent"),
    ).orderBy("session_count", ascending=False).count()


def q3_pyspark():
    from pyspark.sql.functions import round as spark_round, sum as spark_sum, count
    df  = spark.read.parquet(EVENTS_PATH)
    dim = spark.read.parquet(DIMENSION_PATH)
    return df.join(dim, on="page_url", how="left").groupBy(
        "page_section", "country"
    ).agg(
        spark_round(spark_sum("time_spent"), 3).alias("total_time_spent"),
        count("session_id").alias("session_count"),
    ).orderBy("session_count", ascending=False).count()


spark_runs = [
    (q1_pyspark, "Q1_filter_agg"),
    (q2_pyspark, "Q2_explode_groupby"),
    (q3_pyspark, "Q3_join_dimension"),
]


if RUN_VERSION == "compare":
    if not Path(LOCAL_RESULTS_FILE).exists():
        raise FileNotFoundError(f"Brak lokalnych wyników: {LOCAL_RESULTS_FILE}. Uruchom najpierw RUN_VERSION='local'.")
    if not Path(DATAPROC_RESULTS_FILE).exists():
        raise FileNotFoundError(f"Brak wyników dataproc: {DATAPROC_RESULTS_FILE}. Pobierz z GCS: gsutil cp gs://{BUCKET}/project/{DATAPROC_RESULTS_FILE} .")

    df_local    = pd.read_parquet(LOCAL_RESULTS_FILE)
    df_dataproc = pd.read_parquet(DATAPROC_RESULTS_FILE)
    df_all      = pd.concat([df_local, df_dataproc], ignore_index=True)

    print("\n=== WSZYSTKIE WYNIKI ===")
    print(df_all[["library_engine", "query_name", "median_time_s", "peak_memory_mb", "result_check"]].to_string())

    pivot = df_all.pivot_table(
        index="query_name",
        columns="library_engine",
        values=["median_time_s", "peak_memory_mb"],
        aggfunc="median",
    )
    print("\n=== PORÓWNANIE (pivot) ===")
    print(pivot.to_string())

else:
    raise ValueError("RUN_VERSION must be: 'local', 'dataproc', or 'compare'")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 08:03:31 WARN Utils: Your hostname, Viktoriias-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.7 instead (on interface en0)
26/06/11 08:03:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 08:03:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



=== WSZYSTKIE WYNIKI ===
     library_engine          query_name  median_time_s  peak_memory_mb  result_check
0     pyspark_local       Q1_filter_agg       0.168955            0.02             7
1     pyspark_local  Q2_explode_groupby       0.395476            0.05             9
2     pyspark_local   Q3_join_dimension       0.778856            0.05            28
3  pyspark_dataproc       Q1_filter_agg       5.576041            0.02             7
4  pyspark_dataproc  Q2_explode_groupby       5.810479            0.04             9
5  pyspark_dataproc   Q3_join_dimension       4.810305            0.04            28

=== PORÓWNANIE (pivot) ===
                      median_time_s                 peak_memory_mb              
library_engine     pyspark_dataproc pyspark_local pyspark_dataproc pyspark_local
query_name                                                                      
Q1_filter_agg              5.576041      0.168955             0.02          0.02
Q2_explode_groupby         

The whole code used for local and Dataproc execution is in task.py.

As we can see, local PySpark is much faster than Dataproc PySpark, but with every query this difference gets smaller.

The reason why Dataproc get slower for small data is cluster overhead dominates: job submission, YARN/HDFS scheduling, task serialization, and network. For 204 MB of data, the cost of starting the cluster outweighs the benefit of distributed computation.

Why does the difference shrink for Q3? Q3 performs a join between two tables — a shuffle-intensive operation where Dataproc gains relatively more from parallelism, so the ratio drops from x33 to x6.

The results are deterministic — result_check is the same in both environments (7 / 9 / 28), confirming correctness.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [ ]:
# FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
The query that best shows the difference between DataFrame and SQL engines is Q3 (join with a dimension table).
In this query, we join a large fact table with a small dimension table and then do a group-by aggregation.

A SQL engine like DuckDB has a query optimizer. It can see that the dimension table is very small, 
so it automatically uses a broadcast join — it copies the small table to memory and joins it very fast. 
The user does not need to do anything special. DuckDB decides the best join strategy by itself.

A DataFrame engine like Pandas does not have this kind of optimizer. 
It just follows the code step by step. First it merges the full table, then it filters and aggregates. 
This uses more memory and is slower, because Pandas does not "think" about how to make the join better.

So this query is a good test because it shows one important thing: SQL engines can optimize automatically, 
but DataFrame engines cannot. The difference in speed and memory usage between DuckDB and Pandas is very clear in this query, 
which makes it the best example to compare the two types of engines.
"""

display_answer("Final answer 1: Which query best exposes the difference between DataFrame and SQL engines?", FINAL_ANSWER_1)

**Final answer 1: Which query best exposes the difference between DataFrame and SQL engines?**

The query that best shows the difference between DataFrame and SQL engines is Q3 (join with a dimension table).
In this query, we join a large fact table with a small dimension table and then do a group-by aggregation.

A SQL engine like DuckDB has a query optimizer. It can see that the dimension table is very small, 
so it automatically uses a broadcast join — it copies the small table to memory and joins it very fast. 
The user does not need to do anything special. DuckDB decides the best join strategy by itself.

A DataFrame engine like Pandas does not have this kind of optimizer. 
It just follows the code step by step. First it merges the full table, then it filters and aggregates. 
This uses more memory and is slower, because Pandas does not "think" about how to make the join better.

So this query is a good test because it shows one important thing: SQL engines can optimize automatically, 
but DataFrame engines cannot. The difference in speed and memory usage between DuckDB and Pandas is very clear in this query, 
which makes it the best example to compare the two types of engines.

In [ ]:
# FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
The most memory-sensitive query is Q2 (list/tag explode with high-cardinality group-by).

This query takes a column that contains a list of tags and uses explode() to turn each tag into its own row. 
The input has 2,000,000 rows (*in debug mode), but after the explode, the output becomes 3,783,555 rows — almost 2 times more. 
This is what makes the query so memory-demanding: the engine must hold a much larger table in memory than the original input.

From the benchmark results, we can see that even in Polars (which handles this better than Pandas), the Peak RAM usage was between 684 MB and 731 MB, depending on the execution mode.

In Pandas, the memory usage is even worse. Pandas stores the list column as an object dtype, and when it explodes the column, it creates a very large intermediate array in memory — more than any other query.
"""

display_answer("Final answer 2: Which query is most memory-sensitive?", FINAL_ANSWER_2)

**Final answer 2: Which query is most memory-sensitive?**

The most memory-sensitive query is Q2 (list/tag explode with high-cardinality group-by).

This query takes a column that contains a list of tags and uses explode() to turn each tag into its own row. 
The input has 2,000,000 rows (*in debug mode), but after the explode, the output becomes 3,783,555 rows — almost 2 times more. 
This is what makes the query so memory-demanding: the engine must hold a much larger table in memory than the original input.

From the benchmark results, we can see that even in Polars (which handles this better than Pandas), the Peak RAM usage was between 684 MB and 731 MB, depending on the execution mode.

In Pandas, the memory usage is even worse. Pandas stores the list column as an object dtype, and when it explodes the column, it creates a very large intermediate array in memory — more than any other query.

In [ ]:
# FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
For the query used in our experiment (Q2: explode + filter + select), lazy execution did not significantly reduce the amount of data read, but it did reduce runtime.

All modes produced the same number of output rows, which means the same data was materialized. Lazy mode was about 22% faster than eager (*in debug mode), but it actually used a little more peak RAM. This is probably because all modes ran in the same notebook kernel, so old memory allocations from earlier runs affect the measurement.
"""

display_answer("Final answer 3: Did lazy execution change the amount of data read or materialized?", FINAL_ANSWER_3)

**Final answer 3: Did lazy execution change the amount of data read or materialized?**

For the query used in our experiment (Q2: explode + filter + select), lazy execution did not significantly reduce the amount of data read, but it did reduce runtime.

All modes produced the same number of output rows, which means the same data was materialized. Lazy mode was about 22% faster than eager (*in debug mode), but it actually used a little more peak RAM. This is probably because all modes ran in the same notebook kernel, so old memory allocations from earlier runs affect the measurement.

In [ ]:
# FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
Runtime - 100% yes:
- streaming_collect was faster than eager (0.32s vs 0.57s)
- streaming_sink was faster than eager (0.26s vs 0.57s)
This is because streaming processes the data in small batches and pipelines the steps, so it does not wait to load everything before starting to work.

Memory — not sure:
All four modes ran inside the same notebook kernel. The psutil RSS measurement captures the highest memory used by the whole process, including memory from previous runs that was not fully released. 
Because eager ran first and streaming ran later, the later measurements already include leftover allocations from before. 
This makes streaming modes look like they used more memory, which is misleading.

Streaming modes reduced runtime clearly, but the memory results are misleading because of a measurement limitation.
"""

display_answer("Final answer 4: Did streaming collection reduce memory, runtime, or both?", FINAL_ANSWER_4)

**Final answer 4: Did streaming collection reduce memory, runtime, or both?**

Runtime - 100% yes:
- streaming_collect was faster than eager (0.32s vs 0.57s)
- streaming_sink was faster than eager (0.26s vs 0.57s)
This is because streaming processes the data in small batches and pipelines the steps, so it does not wait to load everything before starting to work.

Memory — not sure:
All four modes ran inside the same notebook kernel. The psutil RSS measurement captures the highest memory used by the whole process, including memory from previous runs that was not fully released. 
Because eager ran first and streaming ran later, the later measurements already include leftover allocations from before. 
This makes streaming modes look like they used more memory, which is misleading.

Streaming modes reduced runtime clearly, but the memory results are misleading because of a measurement limitation.

In [ ]:
# FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
It is better in the following scenarios:

1. When Output Volume Exceeds Available Memory (RAM):
   'sink_parquet' is mandatory when the final output dataset is too large to fit into the 
   machine's RAM. While 'collect(engine="streaming")' processes intermediate data in batches, 
   it still attempts a full in-memory materialization of the final DataFrame. This can cause the standard collect to cause a crash of the disk. A streaming sink 
   bypasses RAM completely by writing data chunks directly to disk as they are processed.

2. In Pipeline Checkpointing and ETL Ingestion:
   When writing batch-processing jobs (ETL) where the goal is simply to read, filter/transform, 
   and save the data for down-stream consumers, collecting the data into the local Python 
   session is an expensive and useless overhead. A streaming sink keeps the memory profile 
   flat and predictable, freeing up CPU caches solely for data transformation.

3. Handling High Data-Amplification Query Shapes:
   As seen in our Q2 Explode query, where 2 million initial rows grew into nearly 3.8 million rows, 
   the query shape itself can unexpectedly inflate data volume. A streaming sink acts as a 
   safety net; even if a query causes a massive explosion in row count, the engine will safely 
   spill the data straight into the Parquet file without paralyzing the entire operating system.
"""

display_answer("Final answer 5:  When was a streaming sink more appropriate than collecting the result?", FINAL_ANSWER_5)

**Final answer 5:  When was a streaming sink more appropriate than collecting the result?**

It is better in the following scenarios:

1. When Output Volume Exceeds Available Memory (RAM):
   'sink_parquet' is mandatory when the final output dataset is too large to fit into the 
   machine's RAM. While 'collect(engine="streaming")' processes intermediate data in batches, 
   it still attempts a full in-memory materialization of the final DataFrame. This can cause the standard collect to cause a crash of the disk. A streaming sink 
   bypasses RAM completely by writing data chunks directly to disk as they are processed.

2. In Pipeline Checkpointing and ETL Ingestion:
   When writing batch-processing jobs (ETL) where the goal is simply to read, filter/transform, 
   and save the data for down-stream consumers, collecting the data into the local Python 
   session is an expensive and useless overhead. A streaming sink keeps the memory profile 
   flat and predictable, freeing up CPU caches solely for data transformation.

3. Handling High Data-Amplification Query Shapes:
   As seen in our Q2 Explode query, where 2 million initial rows grew into nearly 3.8 million rows, 
   the query shape itself can unexpectedly inflate data volume. A streaming sink acts as a 
   safety net; even if a query causes a massive explosion in row count, the engine will safely 
   spill the data straight into the Parquet file without paralyzing the entire operating system.

In [ ]:
# FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
Yes, local Apache Spark behaved exactly as expected, demonstrating both its architectural limitations on small-to-medium data scales and its long-term theoretical advantages over single-node engines like Polars.

Based on the measurements, Spark was significantly slower when working on the small provided dataset. Since its a task sutiable for single-node processing, it is therefore understantable and expected that Spark with all the overhead needed to run the query has struggled to keep up with Polars. The framework that Spark creates is in this example an overkill.
"""
display_answer("Final answer 6: Did local Spark behave as expected compared with the single-node engines?", FINAL_ANSWER_6)

**Final answer 6: Did local Spark behave as expected compared with the single-node engines?**

Yes, local Apache Spark behaved exactly as expected, demonstrating both its architectural limitations on small-to-medium data scales and its long-term theoretical advantages over single-node engines like Polars.

Based on the measurements, Spark was significantly slower when working on the small provided dataset. Since its a task sutiable for single-node processing, it is therefore understantable and expected that Spark with all the overhead needed to run the query has struggled to keep up with Polars. The framework that Spark creates is in this example an overkill.

In [ ]:
# FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
Size-wise:
Due to the quick scaling of the datasets that get created during a query, it is necessary to determine maximum size boundaries. A standard computing workstations is between 32-64GB of RAM (16GB for worse machines). This means, that at a maximum, we could allocate around 30-60GB of the RAM for computing the query if using engines like Polars. Since the measurements exposed an approximate linear scaling of `total_ram_alloc = 2 * dataset_size`, it is therefore easy to calculate at which point it will be necessary to move to a cluster approach. If the source dataset was expected to reach more than 16-32GB of memory, it would be instantly necessary to move to a cluster approach.

Shape-wise:
Regardless of the dataset size, it would be necessary to move to a cluster approach if the query contains either or both:
- heavy structural amplification (e.g., exploding high-cardinality nested columns),
- high-cardinality group-by operations combined with large-table joins.

That is because such queries require gigantic hash tables, which will quickly result in memory thrashings or crashes in single-machine-mode. That is why a cluster will tackle such tasks better, since it can horizontally distribute the subtasks across nodes.
"""
display_answer("Final answer 7: At what dataset size or query shape would you move from local processing to a cluster?", FINAL_ANSWER_7)

**Final answer 7: At what dataset size or query shape would you move from local processing to a cluster?**

Size-wise:
Due to the quick scaling of the datasets that get created during a query, it is necessary to determine maximum size boundaries. A standard computing workstations is between 32-64GB of RAM (16GB for worse machines). This means, that at a maximum, we could allocate around 30-60GB of the RAM for computing the query if using engines like Polars. Since the measurements exposed an approximate linear scaling of `total_ram_alloc = 2 * dataset_size`, it is therefore easy to calculate at which point it will be necessary to move to a cluster approach. If the source dataset was expected to reach more than 16-32GB of memory, it would be instantly necessary to move to a cluster approach.

Shape-wise:
Regardless of the dataset size, it would be necessary to move to a cluster approach if the query contains either or both:
- heavy structural amplification (e.g., exploding high-cardinality nested columns),
- high-cardinality group-by operations combined with large-table joins.

That is because such queries require gigantic hash tables, which will quickly result in memory thrashings or crashes in single-machine-mode. That is why a cluster will tackle such tasks better, since it can horizontally distribute the subtasks across nodes.

In [31]:
# FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """

PyArrow dtype backend was much faster and used less memory for data loading and aggregation tasks, 
but it was slower for operations with lists.

For Q1 (filter and aggregation), PyArrow was about x17.7 faster (0.35 seconds compared to 6.18 
seconds) and used almost x7 less memory (205 MB compared to 1421 MB). This happened because 
Arrow stores data in a column format and can read data without copying it. String data stays in 
Arrow memory, so it does not need NumPy object arrays.

For Q2 (explode and groupby on tags list), PyArrow was about x4.5 slower (113.6 seconds 
compared to 25.3 seconds). The explode() function is not well optimized for Arrow list columns in Pandas. 
It uses Python-level iteration, which is slower than NumPy object handling for this type of operation.

For Q3 (join and aggregation), PyArrow was about x1.6 faster (10.5 seconds compared to 16.6 seconds) 
and used about 2.1 times less memory (1246 MB compared to 2619 MB). Join operations work better 
because of Arrow’s efficient memory structure.

There are also differences in data types. The default backend uses NumPy data types, such as object 
for strings and datetime64[us] for dates. The PyArrow backend uses Arrow data types, such as large_string[pyarrow] 
and timestamp[us][pyarrow]. These types are more accurate and work better with other Arrow tools.
"""

display_answer("Final answer 8: How did Pandas default backend compare with the PyArrow dtype backend?", FINAL_ANSWER_8)

**Final answer 8: How did Pandas default backend compare with the PyArrow dtype backend?**

PyArrow dtype backend was much faster and used less memory for data loading and aggregation tasks, 
but it was slower for operations with lists.

For Q1 (filter and aggregation), PyArrow was about x17.7 faster (0.35 seconds compared to 6.18 
seconds) and used almost x7 less memory (205 MB compared to 1421 MB). This happened because 
Arrow stores data in a column format and can read data without copying it. String data stays in 
Arrow memory, so it does not need NumPy object arrays.

For Q2 (explode and groupby on tags list), PyArrow was about x4.5 slower (113.6 seconds 
compared to 25.3 seconds). The explode() function is not well optimized for Arrow list columns in Pandas. 
It uses Python-level iteration, which is slower than NumPy object handling for this type of operation.

For Q3 (join and aggregation), PyArrow was about x1.6 faster (10.5 seconds compared to 16.6 seconds) 
and used about 2.1 times less memory (1246 MB compared to 2619 MB). Join operations work better 
because of Arrow’s efficient memory structure.

There are also differences in data types. The default backend uses NumPy data types, such as object 
for strings and datetime64[us] for dates. The PyArrow backend uses Arrow data types, such as large_string[pyarrow] 
and timestamp[us][pyarrow]. These types are more accurate and work better with other Arrow tools.